# Введение в обработку текста на естественном языке

Материалы:
* Макрушин С.В. Лекция 9: Введение в обработку текста на естественном языке\
* https://realpython.com/nltk-nlp-python/
* https://scikit-learn.org/stable/modules/feature_extraction.html

## Задачи для совместного разбора

In [42]:
from sklearn.feature_extraction.text import CountVectorizer
import pymorphy2

1. Считайте слова из файла `litw-win.txt` и запишите их в список `words`. В заданном предложении исправьте все опечатки, заменив слова с опечатками на ближайшие (в смысле расстояния Левенштейна) к ним слова из списка `words`. Считайте, что в слове есть опечатка, если данное слово не содержится в списке `words`. 

In [ ]:
text = '''с велечайшим усилием выбравшись из потока убегающих людей Кутузов со свитой уменьшевшейся вдвое поехал на звуки выстрелов русских орудий'''

2. Разбейте текст из формулировки задания 1 на слова; проведите стемминг и лемматизацию слов.

3. Преобразуйте предложения из формулировки задания 1 в векторы при помощи `CountVectorizer`.

## Лабораторная работа 9

### Расстояние редактирования

1.1 Загрузите предобработанные описания рецептов из файла `preprocessed_descriptions.csv`. Получите набор уникальных слов `words`, содержащихся в текстах описаний рецептов (воспользуйтесь `word_tokenize` из `nltk`). 

In [5]:

import pandas as pd
import numpy as np
import random
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.metrics.distance import edit_distance
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import cosine

import nltk
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = pd.read_csv('preprocessed_descriptions.csv')
text_col = 'description' if 'description' in df.columns else df.select_dtypes(include=['object']).columns[0]

all_tokens = []
for txt in df[text_col].dropna():
    all_tokens.extend(word_tokenize(str(txt)))

words = list(set(all_tokens))
print(f"{len(words)}")

1.2 Сгенерируйте 5 пар случайно выбранных слов и посчитайте между ними расстояние редактирования.

In [6]:
random_pairs = [(random.choice(words), random.choice(words)) for _ in range(5)]
for i, (w1, w2) in enumerate(random_pairs, 1):
    dist = edit_distance(w1, w2)
    print(f"  Пара {i}: '{w1}' ↔ '{w2}' | Расстояние: {dist}")

NameError: name 'words' is not defined

1.3 Напишите функцию, которая для заданного слова `word` возвращает `k` ближайших к нему слов из списка `words` (близость слов измеряется с помощью расстояния Левенштейна)

In [ ]:
def get_k_nearest_words(word, k, word_list):
    distances = [(w, edit_distance(word, w)) for w in word_list if w != word]
    distances.sort(key=lambda x: x[1])
    return distances[:k]

### Стемминг, лемматизация

2.1 На основе результатов 1.1 создайте `pd.DataFrame` со столбцами: 
    * word
    * stemmed_word 
    * normalized_word 

Столбец `word` укажите в качестве индекса. 

Для стемминга воспользуйтесь `SnowballStemmer`, для нормализации слов - `WordNetLemmatizer`. Сравните результаты стемминга и лемматизации.

In [ ]:
stemmer = SnowballStemmer("russian")  
lemmatizer = WordNetLemmatizer()      

data_for_df = []
for w in words:
    data_for_df.append({
        'word': w,
        'stemmed_word': stemmer.stem(w),
        'normalized_word': lemmatizer.lemmatize(w)
    })

df_words = pd.DataFrame(data_for_df).set_index('word')
print(df_words.head(10))


2.2. Удалите стоп-слова из описаний рецептов. Какую долю об общего количества слов составляли стоп-слова? Сравните топ-10 самых часто употребляемых слов до и после удаления стоп-слов.

In [ ]:
try:
    stop_words = set(stopwords.words('russian'))
except Exception:
    stop_words = set(stopwords.words('english'))

total_before = len(all_tokens)
filtered_tokens = [w for w in all_tokens if w.lower() not in stop_words]
total_after = len(filtered_tokens)
stop_fraction = (total_before - total_after) / total_before if total_before > 0 else 0

print(f"\n2.2 Доля стоп-слов от общего числа: {stop_fraction:.2%}")

top10_before = Counter(all_tokens).most_common(10)
top10_after = Counter(filtered_tokens).most_common(10)

print(top10_before)
print(top10_after)


### Векторное представление текста

3.1 Выберите случайным образом 5 рецептов из набора данных. Представьте описание каждого рецепта в виде числового вектора при помощи `TfidfVectorizer`

In [ ]:
sampled_df = df.sample(n=5, random_state=42)
recipe_names = sampled_df['name'].tolist() if 'name' in sampled_df.columns else [f"Recipe_{i}" for i in sampled_df.index]
recipes_texts = sampled_df[text_col].tolist()

vectorizer = TfidfVectorizer(lowercase=True)
tfidf_matrix = vectorizer.fit_transform(recipes_texts)
print(tfidf_matrix.shape)


3.2 Вычислите близость между каждой парой рецептов, выбранных в задании 3.1, используя косинусное расстояние (`scipy.spatial.distance.cosine`) Результаты оформите в виде таблицы `pd.DataFrame`. В качестве названий строк и столбцов используйте названия рецептов.

In [ ]:
n = len(recipe_names)
dist_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i == j:
            dist_matrix[i, j] = 0.0
        else:
            vec_i = tfidf_matrix[i].toarray().flatten()
            vec_j = tfidf_matrix[j].toarray().flatten()
            dist_matrix[i, j] = cosine(vec_i, vec_j)

df_distances = pd.DataFrame(dist_matrix, index=recipe_names, columns=recipe_names)
print(df_distances.round(4))


3.3 Какие рецепты являются наиболее похожими? Прокомментируйте результат (словами).

In [ ]:
mask = np.ones((n, n), dtype=bool)
np.fill_diagonal(mask, False)
min_idx = np.unravel_index(np.argmin(dist_matrix[mask]), dist_matrix.shape)
r1, r2 = recipe_names[min_idx[0]], recipe_names[min_idx[1]]
min_dist = dist_matrix[min_idx[0], min_idx[1]]

print(f"'{r1}' '{r2}'")
print(min_dist:.4f)